In [0]:
from pyspark.sql.functions import (
    col,
    count,
    sum,
    when,
    min,
    max,
    avg
)

BRONZE_TABLE = "workspace.default.bronze_hvfhv_trips"

df = spark.table(BRONZE_TABLE)

In [0]:
timestamp_quality = df.select(
    count("*").alias("total_rows"),

    sum(
        when(
            col("request_datetime") > col("on_scene_datetime"),
            1
        ).otherwise(0)
    ).alias("request_after_on_scene"),

    sum(
        when(
            col("on_scene_datetime") > col("pickup_datetime"),
            1
        ).otherwise(0)
    ).alias("on_scene_after_pickup"),

    sum(
        when(
            col("pickup_datetime") > col("dropoff_datetime"),
            1
        ).otherwise(0)
    ).alias("pickup_after_dropoff")
)

display(timestamp_quality)

In [0]:
numeric_quality = df.select(
    sum(when(col("trip_miles") < 0, 1).otherwise(0)).alias("negative_trip_miles"),

    sum(when(col("trip_time") < 0, 1).otherwise(0)).alias("negative_trip_time"),

    sum(when(col("base_passenger_fare") < 0, 1).otherwise(0)).alias("negative_base_fare"),

    sum(when(col("tolls") < 0, 1).otherwise(0)).alias("negative_tolls"),

    sum(when(col("bcf") < 0, 1).otherwise(0)).alias("negative_bcf"),

    sum(when(col("sales_tax") < 0, 1).otherwise(0)).alias("negative_sales_tax"),

    sum(when(col("tips") < 0, 1).otherwise(0)).alias("negative_tips"),

    sum(when(col("driver_pay") < 0, 1).otherwise(0)).alias("negative_driver_pay")
)

display(numeric_quality)

In [0]:
flag_columns = [
    "shared_request_flag",
    "shared_match_flag",
    "access_a_ride_flag",
    "wav_request_flag",
    "wav_match_flag"
]

for c in flag_columns:
    print(f"===== {c} =====")

    display(
        df.groupBy(c)
          .count()
          .orderBy(col("count").desc())
    )

In [0]:
display(
    df.groupBy("hvfhs_license_num")
      .count()
      .orderBy(col("count").desc())
)

In [0]:
location_quality = df.select(
    count("*").alias("total_rows"),

    sum(
        when(col("PULocationID").isNull(), 1).otherwise(0)
    ).alias("null_pickup_location"),

    sum(
        when(col("DOLocationID").isNull(), 1).otherwise(0)
    ).alias("null_dropoff_location"),

    min("PULocationID").alias("min_pickup_location"),
    max("PULocationID").alias("max_pickup_location"),

    min("DOLocationID").alias("min_dropoff_location"),
    max("DOLocationID").alias("max_dropoff_location")
)

display(location_quality)

In [0]:
from pyspark.sql.functions import unix_timestamp

df_timing = df.select(
    "trip_time",
    (
        unix_timestamp("dropoff_datetime")
        - unix_timestamp("pickup_datetime")
    ).alias("calculated_trip_time")
)

display(
    df_timing.limit(20)
)

In [0]:
display(
    df.filter(col("base_passenger_fare") < 0)
      .select(
          "hvfhs_license_num",
          "request_datetime",
          "pickup_datetime",
          "dropoff_datetime",
          "trip_miles",
          "trip_time",
          "base_passenger_fare",
          "tolls",
          "bcf",
          "sales_tax",
          "congestion_surcharge",
          "airport_fee",
          "tips",
          "driver_pay"
      )
      .limit(20)
)

In [0]:
display(
    df.filter(col("driver_pay") < 0)
      .select(
          "hvfhs_license_num",
          "request_datetime",
          "pickup_datetime",
          "dropoff_datetime",
          "trip_miles",
          "trip_time",
          "base_passenger_fare",
          "tips",
          "driver_pay"
      )
)

In [0]:
print("Bronze Rows: ", df.count())

In [0]:
display(
    df.filter(
        col("request_datetime") > col("on_scene_datetime")
    )
    .select(
        "hvfhs_license_num",
        "dispatching_base_num",
        "originating_base_num",
        "request_datetime",
        "on_scene_datetime",
        "pickup_datetime",
        "dropoff_datetime",
        "trip_miles",
        "trip_time"
    )
    .limit(20)
)

In [0]:
from pyspark.sql.functions import unix_timestamp, abs

timestamp_difference = (
    unix_timestamp("on_scene_datetime")
    - unix_timestamp("request_datetime")
)

display(
    df.filter(
        col("request_datetime") > col("on_scene_datetime")
    )
    .select(
        abs(timestamp_difference).alias("seconds_difference")
    )
    .summary()
)

In [0]:
from pyspark.sql.functions import (
    col,
    abs,
    unix_timestamp
)

df_time_check = df.select(
    "trip_time",
    (
        unix_timestamp("dropoff_datetime")
        - unix_timestamp("pickup_datetime")
    ).alias("calculated_trip_time_seconds")
)

display(
    df_time_check
    .withColumn(
        "difference_seconds",
        col("trip_time") - col("calculated_trip_time_seconds")
    )
    .select(
        "trip_time",
        "calculated_trip_time_seconds",
        "difference_seconds"
    )
    .summary()
)